In [ ]:
# Core libs
import pandas as pd
import numpy as np

# Modeling
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# Download link helper
from IPython.display import FileLink, display


In [ ]:
train = pd.read_csv("/kaggle/input/titanic/train.csv")
test  = pd.read_csv("/kaggle/input/titanic/test.csv")

print("Train shape:", train.shape)
print("Test shape:",  test.shape)
train.head()


In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Family-based features
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

    # Title from Name
    df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
    df["Title"] = df["Title"].replace(
        ["Lady","Countess","Capt","Col","Don","Dr","Major","Rev","Sir","Jonkheer","Dona"], "Rare"
    )
    df["Title"] = df["Title"].replace({"Mlle":"Miss", "Ms":"Miss", "Mme":"Mrs"})

    return df

train_fe = add_features(train)
test_fe  = add_features(test)


In [ ]:
# Target
y = train_fe["Survived"]

# Features we’ll use
num_features = ["Pclass", "Age", "SibSp", "Parch", "Fare", "FamilySize", "IsAlone"]
cat_features = ["Sex", "Embarked", "Title"]

X = train_fe[num_features + cat_features]
X_test = test_fe[num_features + cat_features]

# Preprocessing: impute nums with median; cats with most_frequent + one-hot
numeric_tf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_tf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_tf, num_features),
        ("cat", categorical_tf, cat_features),
    ]
)


In [ ]:
model = RandomForestClassifier(
    n_estimators=500,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model)
])

# 5-fold stratified CV for a quick quality check
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, scoring="accuracy", cv=cv, n_jobs=-1)
print("CV scores:", np.round(scores, 4))
print("Mean CV:", scores.mean().round(4))


In [ ]:
clf.fit(X, y)
pred = clf.predict(X_test)
print("Predictions made for test rows:", len(pred))


In [ ]:
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": pred.astype(int)
})

# Safety check: must be exactly 418 x 2
assert submission.shape == (418, 2), f"Got {submission.shape}, expected (418, 2)."

submission.to_csv("submission.csv", index=False)
print("✅ Saved /kaggle/working/submission.csv with", len(submission), "rows.")
display(submission.head())
display(submission.tail())

# Clickable download link (appears under the cell output)
display(FileLink("submission.csv"))
